# Final benchmark with the current pipeline (Kaggle)
Qwen2.5-3B, Qwen2.5-7B and Phi-3.5-mini as **writers**; Qwen2.5-7B as the **fixed verifier** for all of them.
16 frozen articles. Uses every fix made since the first runs (Spanish facts, number matching, quote length,
code-built bullets with evidence quotes). Replaces the provisional tables in the README.

**Before running:** Session options -> **GPU T4 x2**, **Internet: On**. About 35-45 min.

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.exists('newsroom-agents'):
    !git clone https://github.com/Juanangara/newsroom-agents.git
%cd /kaggle/working/newsroom-agents
!git pull -q
!git log --oneline | head -1
!pip install -q pydantic feedparser trafilatura googlenewsdecoder bitsandbytes accelerate


In [ ]:
import sys, json, torch
sys.path.insert(0, '/kaggle/working/newsroom-agents/src')
print('GPUs:', torch.cuda.device_count())
from newsroom.benchmark import run_benchmark

MODELS = ['Qwen/Qwen2.5-3B-Instruct', 'Qwen/Qwen2.5-7B-Instruct', 'microsoft/Phi-3.5-mini-instruct']
SNAPSHOTS = ['data/ia_colombia.json', 'data/reforma_pensional.json']
res = run_benchmark(MODELS, SNAPSHOTS, verifier_id='Qwen/Qwen2.5-7B-Instruct',
                    tag='final', out_dir='/kaggle/working/out')


## Sanity checks and diagnostics

In [ ]:
from newsroom.agents import looks_english
for r in res:
    ev = r['evidence']
    print(r['model'].split('/')[-1])
    print('   facts:', len(ev), '| in English:', sum(looks_english(e['fact']) for e in ev),
          '| incomplete discarded:', r['incomplete_facts_discarded'],
          '| editor summaries dropped:', r['editor_summaries_dropped'])
    print('   drop reasons:', r['drop_reasons'])


Download from the **Output** tab: `out/benchmark_final.json` (reports + evidence per fact) and `out/benchmark_final.md` (the table).